# Task 1: Supervised Machine Learning Models

Two supervised models trained on the engineered Metro Interstate Traffic Volume features:

- **Regression** — predicts `traffic_volume`. Algorithms: Linear Regression, Random Forest Regressor. **Metrics**: MAE, R².
- **Classification** — predicts a **proxy accident-risk label**. Algorithms: Logistic Regression, Random Forest Classifier. **Metrics**: Accuracy, Precision, Recall, F1, ROC AUC.

### Proxy accident-risk label
No accident dataset was provided for this capstone. Per the assignment note, a proxy label is built from established real-world accident-risk factors that *are* available here: heavy congestion combined with adverse weather. An hour is flagged `accident_risk = 1` when **both**:
- `traffic_volume` is in the top third of observed volumes (congested), **and**
- the weather that hour is adverse (severe weather, precipitation, or heavy cloud cover / overcast).

This approximates *elevated-risk conditions*, not a record of actual accidents, and is described as such in the final report.

### Leakage note
The proxy label is derived **from** `traffic_volume`, and `traffic_volume` is also the regression target. So `traffic_volume` (and anything derived directly from it — `traffic_volume_zscore`/`minmax`, `congestion_category`) is deliberately **excluded** from the shared feature set below. Without this, both models could trivially "cheat" using a feature that already encodes the answer.

## 1. Imports

In [1]:
import logging

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import (
    mean_absolute_error,
    r2_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
)

RANDOM_STATE = 42

## 2. Logging setup

A named logger (`logging.getLogger("task1_models")`) is used instead of the root logger, with `propagate = False`, so this doesn't collide with Jupyter's own logging setup or double-print.

In [2]:
logger = logging.getLogger("task1_models")


def setup_logging():
    if logger.handlers:
        return  # guard: prevents duplicate handlers (and duplicate log lines) if this cell is re-run
    logger.setLevel(logging.DEBUG)
    fmt = logging.Formatter(
        "%(asctime)s | %(levelname)s | %(name)s | %(message)s",
        datefmt="%Y-%m-%d %H:%M:%S",
    )

    fh = logging.FileHandler("pipeline.log", mode="a", encoding="utf-8")
    fh.setLevel(logging.DEBUG)
    fh.setFormatter(fmt)

    ch = logging.StreamHandler()
    ch.setLevel(logging.INFO)
    ch.setFormatter(fmt)

    logger.addHandler(fh)
    logger.addHandler(ch)
    logger.propagate = False  # don't also hand records to Jupyter's root logger


setup_logging()
logger.info("Logging initialised — DEBUG+ to pipeline.log, INFO+ to this notebook")

2026-09-23 22:22:41 | INFO | task1_models | Logging initialised — DEBUG+ to pipeline.log, INFO+ to this notebook


## 3. Load the feature-engineered dataset



In [3]:
FEATURES_PATH = "Features_Metro_Interstate_Traffic_Volume.csv"

df = pd.read_csv(FEATURES_PATH, parse_dates=["date_time"])
logger.info("Loaded %s rows, %s columns from %s", len(df), df.shape[1], FEATURES_PATH)
df.head()

2026-09-23 22:22:41 | INFO | task1_models | Loaded 48187 rows, 38 columns from Features_Metro_Interstate_Traffic_Volume.csv


,holiday,temp,rain_1h,snow_1h,clouds_all,weather_main,weather_description,date_time,traffic_volume,month,...,is_precipitation,total_precipitation,is_overcast,is_clear,is_severe_weather,temp_c_zscore,traffic_volume_zscore,temp_c_minmax,traffic_volume_minmax,congestion_category
0,NaN,288.28,0.0,0.0,40,Clouds,scattered clouds,2012-10-02 09:00:00,5545,10,...,0,0.0,0,0,0,0.530410,1.150193,0.929726,0.761676,High
1,NaN,289.36,0.0,0.0,75,Clouds,broken clouds,2012-10-02 10:00:00,4516,10,...,0,0.0,0,0,0,0.611377,0.632315,0.933209,0.620330,Medium
2,NaN,289.58,0.0,0.0,90,Clouds,overcast clouds,2012-10-02 11:00:00,4767,10,...,0,0.0,1,0,0,0.627871,0.758639,0.933918,0.654808,High
3,NaN,290.13,0.0,0.0,90,Clouds,overcast clouds,2012-10-02 12:00:00,5026,10,...,0,0.0,1,0,0,0.669104,0.888990,0.935692,0.690385,High
4,NaN,291.14,0.0,0.0,75,Clouds,broken clouds,2012-10-02 13:00:00,4918,10,...,0,0.0,0,0,0,0.744823,0.834635,0.938949,0.675549,High


## 4. Holiday flag and proxy accident-risk label

`is_holiday` is a binary flag built from the text `holiday` column (handles both real `NaN` and the literal `'None'` category the raw dataset uses for non-holiday hours).

In [4]:
is_holiday = df["holiday"].notna() & (df["holiday"] != "None")
df["is_holiday"] = is_holiday.astype(int)
logger.info("is_holiday flag built: %s holiday hours out of %s", int(df['is_holiday'].sum()), len(df))

2026-09-23 22:22:41 | INFO | task1_models | is_holiday flag built: 61 holiday hours out of 48187


In [5]:
congestion_cutoff = df["traffic_volume"].quantile(2 / 3)
is_congested = df["traffic_volume"] >= congestion_cutoff

adverse_cols = ["is_severe_weather", "is_precipitation", "is_overcast"]
missing = [c for c in adverse_cols if c not in df.columns]
if missing:
    logger.error("Missing weather indicator columns %s", missing)
    raise KeyError(
        f"Missing weather indicator columns {missing}. "
        "Run feature_engineering.py (engineer_weather_features) first."
    )

is_adverse_weather = (
    (df["is_severe_weather"] == 1)
    | (df["is_precipitation"] == 1)
    | (df["is_overcast"] == 1)
)

df["accident_risk"] = (is_congested & is_adverse_weather).astype(int)

rate = df["accident_risk"].mean() * 100
logger.info("Congestion cutoff (traffic_volume): %.0f", congestion_cutoff)
logger.info("accident_risk positive rate: %.1f%%", rate)

2026-09-23 22:22:41 | INFO | task1_models | Congestion cutoff (traffic_volume): 4571
2026-09-23 22:22:41 | INFO | task1_models | accident_risk positive rate: 13.6%


## 5. Common, well-engineered feature set

Shared by both models: time-based features (including cyclical `sin`/`cos` encodings of hour and day of week), weather one-hot encodings and derived indicators, and the holiday flag. `traffic_volume` and anything derived from it are excluded (see the leakage note above).

In [12]:
time_features = [
    "hour",
    "day_of_week",
    "is_weekend",
    "hour_sin",
    "hour_cos"
]

exclude_raw_text = {"weather_main", "weather_description"}
weather_onehot = sorted(
    c for c in df.columns if c.startswith("weather_") and c not in exclude_raw_text
)

weather_derived = [
    "is_precipitation",
    "total_precipitation",
    "is_overcast",
    "is_clear",
    "is_severe_weather",
    "clouds_all",
    "temp_c",
]

holiday_feature = ["is_holiday"]

feature_cols = time_features + weather_onehot + weather_derived + holiday_feature
missing = [c for c in feature_cols if c not in df.columns]
if missing:
    logger.error("Missing expected feature columns: %s", missing)
    raise KeyError(
        f"Missing expected feature columns: {missing}. "
    )

logger.info("Using %s features: %s", len(feature_cols), feature_cols)
feature_cols

2026-09-23 22:24:48 | INFO | task1_models | Using 24 features: ['hour', 'day_of_week', 'is_weekend', 'hour_sin', 'hour_cos', 'weather_Clear', 'weather_Clouds', 'weather_Drizzle', 'weather_Fog', 'weather_Haze', 'weather_Mist', 'weather_Rain', 'weather_Smoke', 'weather_Snow', 'weather_Squall', 'weather_Thunderstorm', 'is_precipitation', 'total_precipitation', 'is_overcast', 'is_clear', 'is_severe_weather', 'clouds_all', 'temp_c', 'is_holiday']


['hour',
 'day_of_week',
 'is_weekend',
 'hour_sin',
 'hour_cos',
 'weather_Clear',
 'weather_Clouds',
 'weather_Drizzle',
 'weather_Fog',
 'weather_Haze',
 'weather_Mist',
 'weather_Rain',
 'weather_Smoke',
 'weather_Snow',
 'weather_Squall',
 'weather_Thunderstorm',
 'is_precipitation',
 'total_precipitation',
 'is_overcast',
 'is_clear',
 'is_severe_weather',
 'clouds_all',
 'temp_c',
 'is_holiday']

In [13]:
needed = feature_cols + ["traffic_volume", "accident_risk"]
model_df = df[needed].dropna()
dropped = len(df) - len(model_df)
if dropped:
    logger.warning("Dropped %s rows with missing values before modelling", dropped)
else:
    logger.info("No missing values in modelling columns")
logger.info("Modelling on %s rows, %s features", len(model_df), len(feature_cols))

2026-09-23 22:25:04 | INFO | task1_models | No missing values in modelling columns
2026-09-23 22:25:04 | INFO | task1_models | Modelling on 48187 rows, 24 features


## 6. Regression: predicting `traffic_volume`

Two algorithms — a Linear Regression baseline and a Random Forest Regressor — compared on MAE and R².

In [8]:
X = model_df[feature_cols]
y = model_df["traffic_volume"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

regressors = {
    "Linear Regression": LinearRegression(),
    "Random Forest Regressor": RandomForestRegressor(
        n_estimators=200, max_depth=12, random_state=RANDOM_STATE, n_jobs=-1
    ),
}

reg_rows = []
for name, model in regressors.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    mae = mean_absolute_error(y_test, preds)
    r2 = r2_score(y_test, preds)
    reg_rows.append({"model": name, "MAE": round(mae, 2), "R2": round(r2, 4)})
    logger.info("[Regression] %s \u2014 MAE: %.2f, R2: %.4f", name, mae, r2)

reg_results = pd.DataFrame(reg_rows)
reg_results

2026-09-23 22:22:41 | INFO | task1_models | [Regression] Linear Regression — MAE: 820.40, R2: 0.7219
2026-09-23 22:22:47 | INFO | task1_models | [Regression] Random Forest Regressor — MAE: 263.71, R2: 0.9480


,model,MAE,R2
0,Linear Regression,820.40,0.7219
1,Random Forest Regressor,263.71,0.9480


## 7. Classification: predicting `accident_risk` (proxy label)

Two algorithms — Logistic Regression and a Random Forest Classifier — compared on Accuracy, Precision, Recall, F1, and ROC AUC. Since `accident_risk` is an imbalanced minority class (a realistic property for accident-risk data), both models use `class_weight="balanced"` and the split is stratified.

In [9]:
X = model_df[feature_cols]
y = model_df["accident_risk"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

classifiers = {
    "Logistic Regression": LogisticRegression(max_iter=1000, class_weight="balanced"),
    "Random Forest Classifier": RandomForestClassifier(
        n_estimators=200,
        max_depth=12,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        class_weight="balanced",
    ),
}

clf_rows = []
for name, model in classifiers.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    probs = model.predict_proba(X_test)[:, 1]

    acc = accuracy_score(y_test, preds)
    prec = precision_score(y_test, preds, zero_division=0)
    rec = recall_score(y_test, preds, zero_division=0)
    f1 = f1_score(y_test, preds, zero_division=0)
    auc = roc_auc_score(y_test, probs)

    clf_rows.append(
        {
            "model": name,
            "Accuracy": round(acc, 4),
            "Precision": round(prec, 4),
            "Recall": round(rec, 4),
            "F1": round(f1, 4),
            "ROC_AUC": round(auc, 4),
        }
    )
    logger.info(
        "[Classification] %s \u2014 Acc: %.4f, Prec: %.4f, Rec: %.4f, F1: %.4f, AUC: %.4f",
        name, acc, prec, rec, f1, auc,
    )

clf_results = pd.DataFrame(clf_rows)
clf_results

2026-09-23 22:22:50 | INFO | task1_models | [Classification] Logistic Regression — Acc: 0.9104, Prec: 0.6095, Rec: 0.9544, F1: 0.7439, AUC: 0.9597
2026-09-23 22:22:52 | INFO | task1_models | [Classification] Random Forest Classifier — Acc: 0.9511, Prec: 0.7414, Rec: 0.9856, F1: 0.8462, AUC: 0.9924


,model,Accuracy,Precision,Recall,F1,ROC_AUC
0,Logistic Regression,0.9104,0.6095,0.9544,0.7439,0.9597
1,Random Forest Classifier,0.9511,0.7414,0.9856,0.8462,0.9924


## 8. Summary


- **Regression** — lower MAE and higher R² is better. The Random Forest is expected to outperform Linear Regression if the relationship between features and traffic volume is non-linear (e.g. hour-of-day effects), at the cost of interpretability.
- **Classification** — no single metric tells the whole story for an imbalanced label like this one. Recall matters most if the goal is to catch as many high-risk hours as possible; precision matters if false alarms are costly; ROC AUC gives a threshold-independent view of separability.

Regression (predicting traffic volume):
**MAE: 263.67 vs. 820.47 — Random Forest's average prediction error is roughly a third of Linear Regression's.**
**R²: 0.9480 vs. 0.7219 — Random Forest explains about 95% of the variance in traffic volume; Linear Regression only about 72%**

Classification (predicting accident-risk):
**Classification (predicting accident-risk): Random Forest beat Logistic Regression on all five metrics — accuracy, precision, recall, F1, and ROC AUC (0.992 vs. 0.959)**

In [10]:
print("=== Regression results ===")
print(reg_results.to_string(index=False))

print("\n=== Classification results ===")
print(clf_results.to_string(index=False))

logger.info("Notebook execution complete")

2026-09-23 22:22:52 | INFO | task1_models | Notebook execution complete


=== Regression results ===
                  model    MAE     R2
      Linear Regression 820.40 0.7219
Random Forest Regressor 263.71 0.9480

=== Classification results ===
                   model  Accuracy  Precision  Recall     F1  ROC_AUC
     Logistic Regression    0.9104     0.6095  0.9544 0.7439   0.9597
Random Forest Classifier    0.9511     0.7414  0.9856 0.8462   0.9924


In [20]:
import joblib

joblib.dump(model, 'task1_model.joblib')

# To load it back later:
# loaded_model = joblib.load('task1_model.joblib')

['task1_model.joblib']